# Retail Analytics 101 with PySpark (Databricks Free Edition)

This notebook teaches PySpark basics on Databricks using a small **retail** case study (see the [case study description](https://github.com/Naviden/Introduction-to-Big-Data-and-NoSQL/blob/main/case-studies/retail/README.md)).
You will:
- Load the CSV files of the case study
- Inspect and clean data (nulls, duplicates, outliers)
- Parse JSON, join dimensions, derive metrics
- Aggregate with groupBy and window functions
- Save curated tables as Delta

> **Before you start:** nothing to upload. The first cells read the data directly from the course repository on GitHub.

## Setup

In [ ]:
import os
from pyspark.sql import SparkSession

# On Databricks this returns the session that already exists.
# Locally (pip install pyspark) or on Colab it starts a new one.
spark = SparkSession.builder.getOrCreate()

# display(df) shows an interactive table only on Databricks: elsewhere we use show() instead
if "DATABRICKS_RUNTIME_VERSION" not in os.environ:
    def display(df):
        df.show(20, truncate=False)

from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window

We read the data directly from the GitHub repository of the course, so there is nothing to upload. pandas downloads the CSV and Spark turns it into a DataFrame. This is fine for small files; with real big data Spark would read the files directly from distributed storage (see [Lesson 7](https://github.com/Naviden/Introduction-to-Big-Data-and-NoSQL/blob/main/lessons/07_spark_basics_part1.md) for the other options).

In [ ]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/Naviden/Introduction-to-Big-Data-and-NoSQL/main/data"

def load_csv(path, **kwargs):
    """Read a CSV file of the course repository and return a Spark DataFrame."""
    pdf = pd.read_csv(f"{DATA_URL}/{path}", **kwargs)
    pdf = pdf.astype(object).where(pdf.notna(), None)  # pandas NaN -> Spark null
    return spark.createDataFrame(pdf)

## Load data

In [ ]:
customers = load_csv("retail/customers.csv")
products  = load_csv("retail/products.csv")
stores    = load_csv("retail/stores.csv")
tx_raw    = load_csv("retail/transactions.csv")
returns   = load_csv("retail/returns.csv")

print("Rows -> customers:", customers.count(), "products:", products.count(), "stores:", stores.count(),
      "transactions:", tx_raw.count(), "returns:", returns.count())

## Inspect schema and some rows

In [ ]:
customers.printSchema()
products.printSchema()
stores.printSchema()
tx_raw.printSchema()

display(tx_raw.limit(5))

## Missing values
Before cleaning, let's count how many nulls each column of the transactions has.

In [ ]:
null_counts = tx_raw.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in tx_raw.columns])
display(null_counts)

Only `review_rating` has missing values: not every customer leaves a review. We keep these rows (the sale happened anyway) and simply ignore the nulls when we compute average ratings, which is what `F.avg` does by default.

## Cast types and parse JSON column

In [ ]:
tx = (tx_raw
  .withColumn("transaction_id", F.col("transaction_id").cast("long"))
  .withColumn("store_id",       F.col("store_id").cast("int"))
  .withColumn("customer_id",    F.col("customer_id").cast("int"))
  .withColumn("product_id",     F.col("product_id").cast("int"))
  .withColumn("quantity",       F.col("quantity").cast("int"))
  .withColumn("unit_price",     F.col("unit_price").cast("double"))
  .withColumn("discount_pct",   F.col("discount_pct").cast("double"))
  .withColumn("review_rating",  F.col("review_rating").cast("double"))
  .withColumn("is_returned",    F.col("is_returned").cast("int"))
  .withColumn("pre_tax_amount", F.col("pre_tax_amount").cast("double"))
  .withColumn("tax_rate",       F.col("tax_rate").cast("double"))
  .withColumn("tax_amount",     F.col("tax_amount").cast("double"))
  .withColumn("total_amount",   F.col("total_amount").cast("double"))
  .withColumn("transaction_ts", F.to_timestamp("transaction_ts"))
)

promo_schema = T.StructType([
  T.StructField("coupon",   T.StringType()),
  T.StructField("bundle",   T.BooleanType()),
  T.StructField("campaign", T.StringType())
])

tx = tx.withColumn("promo_meta", F.from_json("promo_meta_json", promo_schema))

display(tx.select("transaction_id", "promo_meta_json", "promo_meta").limit(5))

## Remove duplicate lines and flag outliers

In [ ]:
w = Window.partitionBy("transaction_id", "product_id").orderBy(F.col("transaction_ts").desc_nulls_last())
deduped = tx.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

# Flag implausible quantities (keep them but mark for QA)
cleaned = deduped.withColumn("is_outlier_qty", F.col("quantity") > 20)

print("Original tx rows:", tx_raw.count(), " | After de-dup:", deduped.count())
display(cleaned.filter("is_outlier_qty = true").select("transaction_id", "product_id", "quantity", "total_amount").limit(10))

## Join dimensions and compute derived metrics
`tax_rate` exists both in the transactions and in the products table. We keep only the one of the transaction (the rate actually applied), otherwise the joined table would have two columns with the same name and could not be saved as a table.

In [ ]:
products_t = products.select(
    F.col("product_id").cast("int").alias("product_id"),
    "product_name", "category", "brand", "unit_price_list"
)

customers_t = customers.select(
    F.col("customer_id").cast("int").alias("customer_id"),
    "first_name", "last_name", "email", "signup_date", "age", "loyalty_tier", "is_student", "home_city", "home_region"
)

stores_t = stores.select(
    F.col("store_id").cast("int").alias("store_id"),
    "store_name", "city", "region", "opened_date", "store_type"
)

fact = (cleaned
    .join(products_t, "product_id", "left")
    .join(customers_t, "customer_id", "left")
    .join(stores_t, "store_id", "left")
    .withColumn("year", F.year("transaction_ts"))
    .withColumn("month", F.month("transaction_ts"))
    .withColumn("net_after_discount", (F.col("quantity") * F.col("unit_price") * (1 - F.col("discount_pct"))).cast("double"))
)

display(fact.limit(5))

## Aggregations

In [ ]:
rev_month_cat = (fact
  .groupBy("year", "month", "category")
  .agg(F.round(F.sum("total_amount"), 2).alias("revenue"))
  .orderBy("year", "month", "category"))

top_products = (fact
  .groupBy("product_id", "product_name")
  .agg(F.round(F.sum("total_amount"), 2).alias("revenue"))
  .orderBy(F.desc("revenue"))
  .limit(10))

display(rev_month_cat)
display(top_products)

### Loyalty tier impact
Do customers in higher loyalty tiers spend more per transaction, and are they happier?

In [ ]:
loyalty = (fact
  .groupBy("loyalty_tier")
  .agg(F.countDistinct("customer_id").alias("customers"),
       F.count("*").alias("tx"),
       F.round(F.avg("total_amount"), 2).alias("avg_ticket"),
       F.round(F.avg("review_rating"), 2).alias("avg_rating"))
  .orderBy(F.desc("avg_ticket")))

display(loyalty)

## Window functions: rank largest transactions per store

In [ ]:
w_store = Window.partitionBy("store_id").orderBy(F.desc("total_amount"))
ranked = (fact
  .select("store_id", "transaction_id", "total_amount")
  .withColumn("rank_in_store", F.rank().over(w_store)))

display(ranked.filter("rank_in_store <= 3").orderBy("store_id", "rank_in_store"))

## UDF for price buckets and JSON campaign analysis
A **UDF** (user defined function) lets you run your own Python function on each row. It is flexible, but slower than the built-in functions, because every value has to travel from Spark to Python and back. When a built-in function exists (here `F.when` would do the same job), prefer it.

In [ ]:
from pyspark.sql.types import StringType

@F.udf(StringType())
def price_bucket(x):
    if x is None: return "unknown"
    return "low" if x < 10 else ("mid" if x < 30 else "high")

with_bucket = fact.withColumn("price_bucket", price_bucket("unit_price"))
display(with_bucket.groupBy("price_bucket").count())

campaign_perf = (fact
  .withColumn("campaign", F.col("promo_meta.campaign"))
  .groupBy("campaign").agg(F.count("*").alias("tx"), F.round(F.sum("total_amount"), 2).alias("revenue"))
  .orderBy(F.desc("revenue")))

display(campaign_perf)

## Returns: which categories come back most often?
The `returns` table tells us which transactions were returned and why. We join it with the clean transactions to compute the return rate per category.

In [ ]:
returned_ids = returns.select(F.col("transaction_id").cast("long"), F.col("product_id").cast("int"), "reason").dropDuplicates(["transaction_id", "product_id"])

with_returns = fact.join(returned_ids, ["transaction_id", "product_id"], "left")

return_rate = (with_returns
  .groupBy("category")
  .agg(F.count("*").alias("tx"),
       F.count("reason").alias("returned"))
  .withColumn("return_rate_pct", F.round(100 * F.col("returned") / F.col("tx"), 1))
  .orderBy(F.desc("return_rate_pct")))

display(return_rate)
display(returns.groupBy("reason").count().orderBy(F.desc("count")))

## Save curated tables
On Databricks, `saveAsTable` saves the DataFrame as a **Delta** table in the default schema (`workspace.default` on Free Edition). You can then find it under **Catalog** and query it with SQL. Running locally, the same line saves a Parquet table in a `spark-warehouse` folder.

In [ ]:
fact.write.mode("overwrite").saveAsTable("retail_fact")
rev_month_cat.write.mode("overwrite").saveAsTable("retail_revenue_month_category")

display(spark.sql("SELECT category, ROUND(SUM(total_amount), 2) AS revenue FROM retail_fact GROUP BY category ORDER BY revenue DESC"))

## Exercises
1. Which payment method has the highest average ticket? Does it change between channels?
2. For each customer, compute the number of days between the signup date and their first purchase.
3. Using a window function, compute the 7-day moving average of daily revenue for each store.
4. Rewrite the `price_bucket` UDF using `F.when` and check that you get the same counts.